<a href="https://colab.research.google.com/github/brkrishna/TickerTruth/blob/main/notebooks/adjusted_vs_raw_series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Raw vs Adjusted Price Series

When a company does a stock split or issues bonus shares, the market price drops on
the ex-date — but the shareholder is not worse off. Raw (unadjusted) price series
contain these artificial drops, which corrupt any calculation that looks back across the event:
returns, volatility, momentum, moving averages.

This notebook shows:
- What a raw price series looks like across multiple split/bonus events
- How TickerTruth adjustment factors remove the artificial price discontinuities
- The adjustment formula and how to apply it in pandas

**Data source:** `sample_data/adjustment_factors.parquet`  
Synthetic daily prices are generated here for illustration — in production, apply these
factors to your own EOD price feed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

_cwd = Path.cwd()
SAMPLE_DATA = (
    _cwd / "sample_data"
    if (_cwd / "sample_data").exists()
    else _cwd.parent / "sample_data"
)

adj = pd.read_parquet(SAMPLE_DATA / "adjustment_factors.parquet")
adj["ex_date"] = pd.to_datetime(adj["ex_date"])

print("Adjustment factors loaded:")
print(
    adj[
        [
            "symbol",
            "ex_date",
            "action_type",
            "adjustment_factor",
            "cumulative_adjustment_factor",
        ]
    ].to_string(index=False)
)

## Generate a synthetic RELIANCE raw price series

We model a gradual upward price trend interrupted by two splits.
The raw series shows sharp drops on split dates; the adjusted series is smooth.

In [ ]:
rng = np.random.default_rng(42)

# Daily dates: 2019-01-01 to 2024-12-31
dates = pd.date_range("2019-01-01", "2024-12-31", freq="B")  # business days
n = len(dates)

# Underlying 'true' value: smooth log-normal growth
true_log_returns = rng.normal(0.0003, 0.015, size=n)
true_prices = 1000 * np.exp(np.cumsum(true_log_returns))

# RELIANCE events from adjustment factors
rel_events = adj[adj["symbol"] == "RELIANCE"].sort_values("ex_date")
print("RELIANCE events used:")
print(
    rel_events[
        ["ex_date", "action_type", "adjustment_factor", "cumulative_adjustment_factor"]
    ].to_string(index=False)
)

In [ ]:
# Build raw prices: apply each event's drop on its ex_date
raw_prices = true_prices.copy()
for _, event in rel_events.iterrows():
    mask = dates >= event["ex_date"]
    raw_prices[mask] *= event["adjustment_factor"]  # price drops on ex-date


# Build adjusted prices: multiply raw by the inverse of cumulative factor
# (restating all historical prices to post-all-events equivalent)
def get_cumulative_factor_at(date, events):
    """Return cumulative adjustment factor applicable on a given date."""
    past = events[events["ex_date"] <= date]
    if past.empty:
        return 1.0
    return past.iloc[-1]["cumulative_adjustment_factor"]


cum_factors = np.array([get_cumulative_factor_at(d, rel_events) for d in dates])

# Adjusted = raw / cumulative_factor (restates prices to current-equivalent base)
adjusted_prices = raw_prices / cum_factors

df = pd.DataFrame({"date": dates, "raw": raw_prices, "adjusted": adjusted_prices})
df = df.set_index("date")
print(f"Price range (raw):      INR {df['raw'].min():.0f} – {df['raw'].max():.0f}")
print(
    f"Price range (adjusted): INR {df['adjusted'].min():.0f} – {df['adjusted'].max():.0f}"
)

## Plot: raw vs adjusted

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Raw series
ax1.plot(df.index, df["raw"], color="#e05252", linewidth=1.0, label="Raw price")
for _, ev in rel_events.iterrows():
    ax1.axvline(ev["ex_date"], color="#444", linestyle="--", linewidth=0.8, alpha=0.6)
    ax1.text(
        ev["ex_date"],
        df["raw"].max() * 0.95,
        f" {ev['action_type']}\n {ev['adjustment_factor']:.2f}x",
        fontsize=7,
        color="#444",
    )
ax1.set_ylabel("Price (INR)")
ax1.set_title("RELIANCE — Raw (unadjusted) price series", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Adjusted series
ax2.plot(
    df.index, df["adjusted"], color="#3b82f6", linewidth=1.0, label="Adjusted price"
)
for _, ev in rel_events.iterrows():
    ax2.axvline(ev["ex_date"], color="#444", linestyle="--", linewidth=0.8, alpha=0.6)
ax2.set_ylabel("Price (INR)")
ax2.set_title("RELIANCE — Adjusted price series (discontinuities removed)", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.tight_layout()
plt.show()

## The adjustment formula

To restate any historical price to a current-equivalent basis:

```
adjusted_price(t) = raw_price(t) / cumulative_factor(t)
```

Where `cumulative_factor(t)` is the product of all event-level adjustment factors
for events that occurred **after** date `t`.

Alternatively, TickerTruth provides the `cumulative_adjustment_factor` per event
so you don't need to recompute the chain yourself.

In [ ]:
# Sanity check: compute a 1-year return on a date that straddles an event
# With raw prices the return is distorted; with adjusted prices it is correct.

start = pd.Timestamp("2019-12-31")
end = pd.Timestamp("2020-12-31")  # RELIANCE had a split on 2020-01-01

raw_return = df.loc[end, "raw"] / df.loc[start, "raw"] - 1
adj_return = df.loc[end, "adjusted"] / df.loc[start, "adjusted"] - 1

print(
    f"1-year return 2020 (raw):      {raw_return:+.1%}  ← includes split-day price drop"
)
print(f"1-year return 2020 (adjusted): {adj_return:+.1%}  ← true economic return")

---
## Key takeaway

| | Raw series | Adjusted series |
|---|---|---|
| Split/bonus dates | Artificial price drop | Smooth continuation |
| Return calculations straddling events | Wrong | Correct |
| Momentum signals | Spurious negative signal on ex-date | Accurate |
| Volatility estimates | Inflated on event dates | Unbiased |

Always use adjusted prices for any lookback calculation. Use raw prices only
when you need the actual traded price on a specific date.